# TP 2 — Process mining avec PM4Py

## Présentation du contexte
Ce notebook analyse un journal de demandes de congé avec PM4Py.

## Objectifs
- Explorer le journal d'événements.
- Découvrir le processus réel.
- Vérifier la conformité avec un modèle BPMN de référence.
- Analyser la performance.

## Fichiers nécessaires
- `data/demande_conge.csv`
- `data/modele_reference_conge.bpmn`

## Instructions d’exécution
- Exécuter toutes les cellules de haut en bas après redémarrage du noyau.
- Aucun chemin absolu local n'est utilisé.


## 0. Préparation de l’environnement
### Objectif
Initialiser l'environnement, détecter la racine du projet, vérifier les fichiers d'entrée et Graphviz.


In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import sys

import pandas as pd
import pm4py

print(f"Python: {sys.version.split()[0]}")
print(f"PM4Py: {getattr(pm4py, '__version__', 'version indisponible')}")

In [ ]:
def find_project_root(start: Path) -> Path:
    markers = ["pyproject.toml", ".git"]
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(
        "Racine du projet introuvable. Lancez le notebook depuis la racine du dépôt ou depuis notebooks/."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "demande_conge.csv"
BPMN_REFERENCE_PATH = DATA_DIR / "modele_reference_conge.bpmn"

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"CSV_PATH = {CSV_PATH}")
print(f"BPMN_REFERENCE_PATH = {BPMN_REFERENCE_PATH}")
print(f"FIGURES_DIR = {FIGURES_DIR}")


In [ ]:
required_files = [CSV_PATH, BPMN_REFERENCE_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]

dot_path = shutil.which("dot")
if dot_path:
    print(f"Graphviz (dot) détecté: {dot_path}")
else:
    print("Graphviz système (dot) introuvable. Les visualisations Graphviz ne seront pas exportées.")
    print("Installez Graphviz système puis relancez.")

if missing_files:
    print("\nFichiers manquants :")
    for path in missing_files:
        print(f"- {path}")
    print("\nAucune donnée fictive n'est utilisée : les analyses dépendantes des fichiers sont bloquées.")

DATA_READY = len(missing_files) == 0
print(f"\nDATA_READY = {DATA_READY}")


### (Optionnel) Installation Colab
Cette cellule est facultative et n'est pas exécutée automatiquement.


In [ ]:
# Optionnel (Google Colab uniquement)
# !pip install -r requirements.txt
# !apt-get update && apt-get install -y graphviz

## Partie A — Charger et explorer le journal
### A.1 — Préparer l’environnement
#### Objectif
Charger le CSV, valider sa qualité, puis préparer un DataFrame compatible PM4Py.


In [ ]:
REQUIRED_COLUMNS = ["case_id", "activity", "timestamp", "resource"]

if DATA_READY:
    raw_df = pd.read_csv(CSV_PATH, dtype={"case_id": "string"})
    print(f"Lignes chargées: {len(raw_df)}")
else:
    raw_df = None
    print("Données absentes: impossible de charger le journal.")


### A.2 — Charger et formater le journal
#### Objectif
Contrôler les colonnes obligatoires, valeurs manquantes, timestamps invalides, doublons exacts et ordre chronologique.


In [ ]:
if DATA_READY:
    missing_columns = [col for col in REQUIRED_COLUMNS if col not in raw_df.columns]
    if missing_columns:
        raise ValueError(f"Colonnes obligatoires manquantes: {missing_columns}")

    quality_summary = {}
    quality_summary["valeurs_manquantes"] = raw_df[REQUIRED_COLUMNS].isna().sum().to_dict()

    parsed_timestamps = pd.to_datetime(raw_df["timestamp"], errors="coerce", utc=True)
    invalid_timestamps = int(parsed_timestamps.isna().sum())
    quality_summary["timestamps_invalides"] = invalid_timestamps

    exact_duplicates = int(raw_df.duplicated().sum())
    quality_summary["doublons_exacts"] = exact_duplicates

    print("Résumé qualité:")
    print(quality_summary)

    if invalid_timestamps > 0:
        raise ValueError("Des timestamps invalides ont été détectés. Corrigez les données avant l'analyse.")

    journal_df = raw_df.copy()
    journal_df["case_id"] = journal_df["case_id"].astype("string")
    journal_df["timestamp"] = parsed_timestamps
    journal_df["_original_order"] = range(len(journal_df))

    journal_df = journal_df.sort_values(["case_id", "timestamp", "_original_order"], kind="mergesort").reset_index(drop=True)

    same_ts_counts = (
        journal_df.groupby(["case_id", "timestamp"]).size().reset_index(name="n").query("n > 1")
    )
    if not same_ts_counts.empty:
        print("Attention: des événements partagent le même timestamp dans un même cas. L'ordre peut influencer les variantes.")

    event_log_df = pm4py.format_dataframe(
        journal_df.copy(),
        case_id="case_id",
        activity_key="activity",
        timestamp_key="timestamp",
    )

    event_log_df = event_log_df.sort_values(
        ["case:concept:name", "time:timestamp", "_original_order"], kind="mergesort"
    ).reset_index(drop=True)

    print("DataFrame PM4Py prêt.")
else:
    journal_df = None
    event_log_df = None


### A.3 — Statistiques de base
#### Objectif
Mesurer la taille du journal et la fréquence des activités.


In [ ]:
if DATA_READY:
    nb_cases = int(event_log_df["case:concept:name"].nunique())
    nb_events = int(len(event_log_df))
    nb_activities = int(event_log_df["concept:name"].nunique())

    activity_freq = (
        event_log_df["concept:name"].value_counts(dropna=False)
        .rename_axis("activité")
        .reset_index(name="fréquence")
    )

    stats_q1 = {
        "nombre_de_cas": nb_cases,
        "nombre_d_evenements": nb_events,
        "nombre_d_activites_distinctes": nb_activities,
    }

    print(stats_q1)
    display(activity_freq)
else:
    stats_q1 = {}
    activity_freq = pd.DataFrame()
    print("Statistiques indisponibles: données manquantes.")


### A.4 — Variantes
#### Objectif
Calculer le nombre de variantes et les cinq plus fréquentes, puis la proportion des deux premières.


In [ ]:
def normalize_variants(variants_result):
    normalized = []

    if isinstance(variants_result, dict):
        iterator = variants_result.items()
    else:
        iterator = variants_result

    for item in iterator:
        if isinstance(item, tuple) and len(item) == 2:
            variant, value = item
        else:
            variant, value = item, None

        if isinstance(variant, (list, tuple)):
            variant_key = tuple(str(x) for x in variant)
        else:
            variant_key = (str(variant),)

        if isinstance(value, int):
            count = value
        elif isinstance(value, (list, tuple, set)):
            count = len(value)
        elif value is None:
            count = 1
        else:
            try:
                count = int(value)
            except Exception:
                count = 1

        normalized.append((variant_key, count))

    return normalized

if DATA_READY:
    if hasattr(pm4py, "get_variants_as_tuples"):
        variants_raw = pm4py.get_variants_as_tuples(event_log_df)
    else:
        variants_raw = pm4py.get_variants(event_log_df)

    variants_normalized = normalize_variants(variants_raw)
    variants_sorted = sorted(variants_normalized, key=lambda x: x[1], reverse=True)

    nb_variants = len(variants_sorted)
    top5_variants = variants_sorted[:5]

    top_two_count = sum(count for _, count in variants_sorted[:2])
    top_two_ratio = (top_two_count / len(event_log_df["case:concept:name"].unique()) * 100) if variants_sorted else 0

    variants_df = pd.DataFrame(
        [
            {
                "rang": i + 1,
                "variante": " > ".join(variant),
                "effectif": count,
            }
            for i, (variant, count) in enumerate(top5_variants)
        ]
    )

    variants_summary = {
        "nombre_de_variantes": nb_variants,
        "effectif_2_premieres": top_two_count,
        "proportion_2_premieres_pct": round(top_two_ratio, 2),
    }

    print(variants_summary)
    display(variants_df)
else:
    variants_sorted = []
    variants_summary = {}
    print("Variantes indisponibles: données manquantes.")


## Partie B — Découvrir le processus réel
### B.1 — Graphe des enchaînements
#### Objectif
Découvrir et exporter le DFG de fréquence observé.


In [ ]:
if DATA_READY:
    from pm4py.visualization.dfg import visualizer as dfg_visualizer

    dfg_freq, start_activities, end_activities = pm4py.discover_dfg(event_log_df)

    if shutil.which("dot"):
        dfg_freq_path = FIGURES_DIR / "dfg_frequence.png"
        gviz_freq = dfg_visualizer.apply(
            dfg_freq,
            log=event_log_df,
            variant=dfg_visualizer.Variants.FREQUENCY,
            parameters={
                dfg_visualizer.Variants.FREQUENCY.value.Parameters.START_ACTIVITIES: start_activities,
                dfg_visualizer.Variants.FREQUENCY.value.Parameters.END_ACTIVITIES: end_activities,
            },
        )
        dfg_visualizer.save(gviz_freq, str(dfg_freq_path))
        print(f"DFG fréquence sauvegardé: {dfg_freq_path}")
    else:
        print("Export DFG ignoré: Graphviz système absent.")
else:
    dfg_freq = None
    print("DFG indisponible: données manquantes.")


### B.2 — BPMN découvert avec filtrage
#### Objectif
Découvrir un modèle BPMN avec `noise_threshold=0.2`.


In [ ]:
if DATA_READY:
    bpmn_filtered = pm4py.discover_bpmn_inductive(event_log_df, noise_threshold=0.2)

    if hasattr(pm4py, "write_bpmn"):
        filtered_bpmn_path = DATA_DIR / "modele_decouvert_filtre.bpmn"
        pm4py.write_bpmn(bpmn_filtered, str(filtered_bpmn_path))
        print(f"BPMN filtré sauvegardé: {filtered_bpmn_path}")

    if shutil.which("dot"):
        from pm4py.visualization.bpmn import visualizer as bpmn_visualizer

        filtered_fig_path = FIGURES_DIR / "bpmn_decouvert_filtre.png"
        gviz_filtered = bpmn_visualizer.apply(bpmn_filtered)
        bpmn_visualizer.save(gviz_filtered, str(filtered_fig_path))
        print(f"Figure BPMN filtré: {filtered_fig_path}")
else:
    bpmn_filtered = None
    print("BPMN filtré indisponible: données manquantes.")


### B.3 — BPMN découvert sans filtrage
#### Objectif
Découvrir un modèle BPMN avec `noise_threshold=0.0`.


In [ ]:
if DATA_READY:
    bpmn_unfiltered = pm4py.discover_bpmn_inductive(event_log_df, noise_threshold=0.0)

    if hasattr(pm4py, "write_bpmn"):
        unfiltered_bpmn_path = DATA_DIR / "modele_decouvert_complet.bpmn"
        pm4py.write_bpmn(bpmn_unfiltered, str(unfiltered_bpmn_path))
        print(f"BPMN non filtré sauvegardé: {unfiltered_bpmn_path}")

    if shutil.which("dot"):
        from pm4py.visualization.bpmn import visualizer as bpmn_visualizer

        unfiltered_fig_path = FIGURES_DIR / "bpmn_decouvert_sans_filtre.png"
        gviz_unfiltered = bpmn_visualizer.apply(bpmn_unfiltered)
        bpmn_visualizer.save(gviz_unfiltered, str(unfiltered_fig_path))
        print(f"Figure BPMN non filtré: {unfiltered_fig_path}")
else:
    bpmn_unfiltered = None
    print("BPMN non filtré indisponible: données manquantes.")


### B.4 — Comparaison des modèles
#### Objectif
Comparer les modèles obtenus sans sur-interpréter le filtrage.


In [ ]:
def summarize_bpmn(model):
    if model is None:
        return {"noeuds": None, "flux": None}

    nodes = len(model.get_nodes()) if hasattr(model, "get_nodes") else None
    flows = len(model.get_flows()) if hasattr(model, "get_flows") else None
    return {"noeuds": nodes, "flux": flows}

if DATA_READY:
    comparaison_bpmn = pd.DataFrame(
        [
            {"modele": "filtré (0.2)", **summarize_bpmn(bpmn_filtered)},
            {"modele": "sans filtrage (0.0)", **summarize_bpmn(bpmn_unfiltered)},
        ]
    )
    display(comparaison_bpmn)
    print("Comparer aussi les traces: le modèle découvert décrit un comportement autorisé, pas seulement observé.")
else:
    print("Comparaison indisponible: données manquantes.")


## Partie C — Vérifier la conformité
### C.1 — Charger le modèle de référence
#### Objectif
Charger le BPMN de référence et le convertir en réseau de Petri.


In [ ]:
if DATA_READY:
    bpmn_reference = pm4py.read_bpmn(str(BPMN_REFERENCE_PATH))
    petri_net, initial_marking, final_marking = pm4py.convert_to_petri_net(bpmn_reference)
    print("Modèle de référence chargé et converti.")
else:
    bpmn_reference = None
    petri_net = initial_marking = final_marking = None
    print("Conformité indisponible: données manquantes.")


### C.2 — Fitness et précision
#### Objectif
Calculer fitness, précision et part de traces parfaitement conformes.


In [ ]:
if DATA_READY:
    fitness_result = pm4py.fitness_alignments(event_log_df, petri_net, initial_marking, final_marking)
    precision_value = pm4py.precision_alignments(event_log_df, petri_net, initial_marking, final_marking)

    alignments = pm4py.conformance_diagnostics_alignments(event_log_df, petri_net, initial_marking, final_marking)

    case_ids_order = (
        event_log_df[["case:concept:name"]]
        .drop_duplicates()
        ["case:concept:name"]
        .astype(str)
        .tolist()
    )

    perfect_count = 0
    for al in alignments:
        al_fitness = float(al.get("fitness", 0))
        if al_fitness >= 0.999999:
            perfect_count += 1

    total_cases = len(case_ids_order)
    perfect_pct = (perfect_count / total_cases * 100) if total_cases else 0

    conformite_resume = {
        "fitness": fitness_result.get("log_fitness", fitness_result),
        "precision": precision_value,
        "cas_parfaitement_conformes": perfect_count,
        "pourcentage_cas_parfaitement_conformes": round(perfect_pct, 2),
    }

    print(conformite_resume)
else:
    alignments = []
    conformite_resume = {}
    case_ids_order = []
    print("Mesures de conformité indisponibles: données manquantes.")


### C.3 — Types d’écarts
#### Objectif
Compter les mouvements visibles « En trop » et « Manquant » à partir des alignements calculés une seule fois.


In [ ]:
from collections import Counter

def extract_label(obj):
    if obj is None:
        return None
    if isinstance(obj, str):
        return obj
    if isinstance(obj, (list, tuple)):
        labels = [extract_label(x) for x in obj]
        labels = [x for x in labels if x is not None]
        for label in labels:
            if label != ">>":
                return label
        if labels:
            return labels[0]
        return None
    return str(obj)

def is_silent_model_label(label):
    if label is None:
        return True
    label_str = str(label).strip().lower()
    return label_str in {"tau", "none", ""} or label_str.startswith("tau")

if DATA_READY:
    ecarts = Counter()
    cases_with_extra = set()
    cases_with_missing = set()

    for idx, al in enumerate(alignments):
        case_id = case_ids_order[idx] if idx < len(case_ids_order) else f"case_{idx}"
        for move in al.get("alignment", []):
            if not isinstance(move, (list, tuple)) or len(move) < 2:
                continue

            log_label = extract_label(move[0])
            model_label = extract_label(move[1])

            log_is_skip = (log_label == ">>")
            model_is_skip = (model_label == ">>")

            if model_is_skip and not log_is_skip and log_label not in (None, ">>"):
                ecarts[("En trop", str(log_label))] += 1
                cases_with_extra.add(case_id)

            if log_is_skip and not model_is_skip and not is_silent_model_label(model_label):
                ecarts[("Manquant", str(model_label))] += 1
                cases_with_missing.add(case_id)

    ecarts_df = pd.DataFrame(
        [
            {"type": t, "activite": a, "occurrences": n}
            for (t, a), n in ecarts.items()
        ]
    ).sort_values("occurrences", ascending=False, ignore_index=True)

    print({
        "cas_avec_ecarts_en_trop": len(cases_with_extra),
        "cas_avec_ecarts_manquants": len(cases_with_missing),
    })
    display(ecarts_df)
else:
    ecarts_df = pd.DataFrame()
    print("Types d'écarts indisponibles: données manquantes.")


### C.4 — Congés enregistrés sans validation
#### Objectif
Identifier les cas qui contiennent « Enregistrer le congé » sans aucune « Valider la demande ».


In [ ]:
if DATA_READY:
    activities_by_case = event_log_df.groupby("case:concept:name")["concept:name"].agg(list)

    cases_enreg_sans_validation = [
        case_id
        for case_id, acts in activities_by_case.items()
        if ("Enregistrer le congé" in acts) and ("Valider la demande" not in acts)
    ]

    nb_cases_total = event_log_df["case:concept:name"].nunique()
    nb_sans_validation = len(cases_enreg_sans_validation)
    ratio_sans_validation = (nb_sans_validation / nb_cases_total * 100) if nb_cases_total else 0

    print(f"Nombre de cas: {nb_sans_validation}")
    print(f"Proportion: {ratio_sans_validation:.2f}%")
    print("10 premiers identifiants:")
    print(cases_enreg_sans_validation[:10])

    print("Note: ce contrôle suppose que le journal est suffisamment complet.")
    print("Un autre contrôle utile consiste à détecter les validations tardives (après enregistrement).")
else:
    print("Contrôle indisponible: données manquantes.")


## Partie D — Analyser la performance
### D.1 — Durée des demandes
#### Objectif
Calculer moyenne, médiane, maximum et comparaison à l'objectif de 2 jours.


In [ ]:
if DATA_READY:
    case_time_bounds = event_log_df.groupby("case:concept:name")["time:timestamp"].agg(["min", "max"])
    case_time_bounds["duree_jours"] = (case_time_bounds["max"] - case_time_bounds["min"]).dt.total_seconds() / 86400

    perf_summary = {
        "moyenne_jours": round(float(case_time_bounds["duree_jours"].mean()), 4),
        "mediane_jours": round(float(case_time_bounds["duree_jours"].median()), 4),
        "maximum_jours": round(float(case_time_bounds["duree_jours"].max()), 4),
        "proportion_leq_2_jours_pct": round(float((case_time_bounds["duree_jours"] <= 2).mean() * 100), 2),
    }

    print(perf_summary)
    print("Les durées calculées ici sont calendaires (pas des jours ouvrés).")
else:
    case_time_bounds = pd.DataFrame()
    perf_summary = {}
    print("Durées indisponibles: données manquantes.")


### D.2 — Graphe de performance
#### Objectif
Exporter le DFG de performance et un tableau de délais moyens par transition.


In [ ]:
if DATA_READY:
    from pm4py.visualization.dfg import visualizer as dfg_visualizer

    performance_dfg, perf_start, perf_end = pm4py.discover_performance_dfg(event_log_df)

    transition_perf_rows = []
    for (src, tgt), value in performance_dfg.items():
        if isinstance(value, dict):
            mean_seconds = value.get("mean")
        else:
            mean_seconds = value
        if mean_seconds is None:
            continue
        transition_perf_rows.append({
            "source": src,
            "cible": tgt,
            "delai_moyen_heures": round(float(mean_seconds) / 3600, 2),
        })

    transition_perf_df = pd.DataFrame(transition_perf_rows).sort_values("delai_moyen_heures", ascending=False, ignore_index=True)
    display(transition_perf_df)

    if shutil.which("dot"):
        perf_path = FIGURES_DIR / "dfg_performance.png"
        gviz_perf = dfg_visualizer.apply(
            performance_dfg,
            log=event_log_df,
            variant=dfg_visualizer.Variants.PERFORMANCE,
            parameters={
                dfg_visualizer.Variants.PERFORMANCE.value.Parameters.START_ACTIVITIES: perf_start,
                dfg_visualizer.Variants.PERFORMANCE.value.Parameters.END_ACTIVITIES: perf_end,
            },
        )
        dfg_visualizer.save(gviz_perf, str(perf_path))
        print(f"DFG performance sauvegardé: {perf_path}")
    else:
        print("Export DFG performance ignoré: Graphviz système absent.")
else:
    transition_perf_df = pd.DataFrame()
    print("Performance DFG indisponible: données manquantes.")


### D.3 — Attente avant validation par manager
#### Objectif
Mesurer le délai entre chaque validation et l'événement immédiatement précédent du même cas.


In [ ]:
if DATA_READY:
    ordered = event_log_df.sort_values(
        ["case:concept:name", "time:timestamp", "_original_order"], kind="mergesort"
    ).copy()

    ordered["prev_timestamp"] = ordered.groupby("case:concept:name")["time:timestamp"].shift(1)
    ordered["delai_depuis_evenement_precedent_heures"] = (
        (ordered["time:timestamp"] - ordered["prev_timestamp"]).dt.total_seconds() / 3600
    )

    validations = ordered[ordered["concept:name"] == "Valider la demande"].copy()
    validations = validations[validations["resource"].notna()].copy()

    manager_wait_df = (
        validations.groupby("resource")["delai_depuis_evenement_precedent_heures"]
        .agg(nombre_de_validations="count", delai_moyen_heures="mean")
        .reset_index()
        .rename(columns={"resource": "manager"})
        .sort_values("delai_moyen_heures", ascending=False, ignore_index=True)
    )

    manager_wait_df["delai_moyen_heures"] = manager_wait_df["delai_moyen_heures"].round(2)
    display(manager_wait_df)

    print("Interprétation: ce délai n'est pas le délai total depuis le dépôt, et un cas peut contenir plusieurs validations.")
    print("Les différences entre managers ne prouvent pas, à elles seules, une cause individuelle.")
else:
    manager_wait_df = pd.DataFrame()
    print("Analyse managers indisponible: données manquantes.")


## Synthèse — Réponses aux questions Q1 à Q5
### Objectif
Consolider les réponses en distinguant résultats recalculés et valeurs de référence communiquées.


In [ ]:
if DATA_READY:
    print("Q1 — Statistiques du journal")
    print(stats_q1)
    print(variants_summary)

    print("\nQ2 — Deux variantes les plus fréquentes")
    print("Se référer au tableau des variantes (A.4).")

    print("\nQ3 — Fitness et conformité")
    print(conformite_resume)

    print("\nQ4 — Écarts")
    if not ecarts_df.empty:
        display(ecarts_df.head(10))

    print("\nQ5 — Performance")
    print(perf_summary)
else:
    print("Valeur communiquée lors d’une exécution précédente, non recalculée dans cette session.")


## Conclusion et limites
- Les résultats dépendent de la complétude du journal et de la qualité des timestamps.
- Le filtrage influence la structure des modèles découverts.
- Les alignements distinguent conformité globale et écarts locaux, mais restent sensibles au modèle de référence.
- Les indicateurs temporels présentés ici sont descriptifs et ne prouvent pas une causalité.
